In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import statistics

# Labels

In [ ]:
coding_labels = [
    'transcript_ablation',
    'splice_acceptor_variant',
    'splice_donor_variant',
    'stop_gained',
    'frameshift_variant',
    'stop_lost',
    'start_lost',
    'initiator_codon_variant',
    'transcript_amplification',
    'inframe_insertion',
    'inframe_deletion',
    'missense_variant',
    'protein_altering_variant',
    'splice_donor_5th_base_variant',
    'splice_region_variant',
    'splice_donor_region_variant',
    'splice_polypyrimidine_tract_variant',
    'incomplete_terminal_codon_variant',
    'start_retained_variant',
    'stop_retained_variant',
    'synonymous_variant',
    'coding_sequence_variant',
    'coding_transcript_variant'
]

non_coding_labels = [
    'mature_miRNA_variant',
    '5_prime_UTR_variant',
    '3_prime_UTR_variant',
    'non_coding_transcript_exon_variant',
    'non_coding_exon_variant',
    'intron_variant',
    'NMD_transcript_variant',
    'non_coding_transcript_variant',
    'nc_transcript_variant',
    'upstream_gene_variant',
    'downstream_gene_variant',
    'TFBS_ablation',
    'TFBS_amplification',
    'TF_binding_site_variant',
    'regulatory_region_ablation',
    'regulatory_region_amplification',
    'feature_elongation',
    'regulatory_region_variant',
    'feature_truncation',
    'intergenic_variant',
    'sequence_variant',
]

lof_labels = [
    'transcript_ablation',
    'splice_acceptor_variant',
    'splice_donor_variant',
    'stop_gained',
    'frameshift_variant'
]

# Read data

In [ ]:
vep_df = pd.read_csv(f'/mnt/project/DNM/trios/out/trios_diffs_vep.tsv.bgz', sep = '\t', compression = 'gzip')

vep_df.head()

In [ ]:
dnms = {}

for _, row in vep_df.iterrows():

    dnm = f'chr{row['f0']}:{row['f1']}:{row['f2']}:{row['f3']}'
    vep = json.loads(row['vep'])
    
    dnms[dnm] = {}
                
    if vep['transcript_consequences']:
        t = vep['transcript_consequences'][0]
        dnms[dnm]['consequence'] = t['consequence_terms'][0]
        dnms[dnm]['gene_symbol'] = t['gene_symbol']
        dnms[dnm]['gene_id'] = t['gene_id']
        dnms[dnm]['lof'] = t['lof']
        
    elif vep['intergenic_consequences']:
        i = vep['intergenic_consequences'][0]
        dnms[dnm]['consequence'] = i['consequence_terms'][0]
        
    elif vep['regulatory_feature_consequences']:
        r = vep['regulatory_feature_consequences'][0]
        dnms[dnm]['consequence'] = r['consequence_terms'][0]
      
    elif vep['motif_feature_consequences']:
        m = vep['motif_feature_consequences'][0]
        dnms[dnm]['consequence'] = m['consequence_terms'][0]

# Plot consequences

In [ ]:
dnms_cons = {}

for dnm in dnms:
    if dnms[dnm]['consequence'] not in dnms_cons:
        dnms_cons[dnms[dnm]['consequence']] = 0
    dnms_cons[dnms[dnm]['consequence']] += 1
        
dnms_cons = {k: v for k, v in sorted(dnms_cons.items(), key = lambda item: item[1], reverse = True)}

dnms_cons

In [ ]:
with open('trios_diffs_consequences.txt', 'w') as f:
    f.write('Consequence\tCount\n')
    for cons in dnms_cons:
        f.write(f'{cons}\t{dnms_cons[cons]}\n')

In [ ]:
!dx upload 'trios_diffs_consequences.txt' --path='DNM/trios/out/'

In [ ]:
total = sum(dnms_cons.values())

labels = [f'{label}: {int((count/total)*100)}%' for label, count in dnms_cons.items()]

pie, _ = plt.pie(dnms_cons.values())
plt.legend(pie, labels, loc = 'best', bbox_to_anchor = (1, 1))
plt.show()

In [ ]:
dnms_cons_coding = {}

for label in dnms_cons:
    if label in coding_labels:
        dnms_cons_coding[label] = dnms_cons[label]
        
total_coding = sum(dnms_cons_coding.values())

labels = [f'{label}: {count/total_coding*100:.0f}%' for label, count in dnms_cons_coding.items()]    

pie, _ = plt.pie(dnms_cons_coding.values())
plt.legend(pie, labels, loc = 'best', bbox_to_anchor = (1, 1))
plt.show()

In [ ]:
print(total, total_coding, round(total_coding/total, 4))

# Loss-of-function variants

In [ ]:
lofs = []

for dnm in dnms:
    if dnms[dnm]['consequence'] in lof_labels:
        lofs.append(dnm)
        
print(total, len(lofs), round(len(lofs)/total, 4))
print(total_coding, len(lofs), round(len(lofs)/total_coding, 4))

In [ ]:
gene_hs = {}
genebayes_df = pd.read_csv('s_het_estimates.genebayes.tsv', sep = '\t')
gene_hs = dict(zip(genebayes_df['ensg'], genebayes_df['post_mean']))

In [ ]:
hs = []

for lof in lofs:
    if dnms[lof]['gene_id'] in gene_hs:
        hs.append(gene_hs[dnms[lof]['gene_id']])
        
print(round(min(hs), 4), round(max(hs), 4), round(statistics.mean(hs), 4), 
      round(statistics.median(hs), 4), sum(hs), len(hs))

In [ ]:
plt.boxplot(hs)
plt.ylabel(r'GeneBayes estimate of $s_{het}$')
plt.show()

In [ ]:
with open('trios_diffs_hs.txt', 'w') as f:
    for h in hs:
        f.write(f'{h}\n')

In [ ]:
!dx upload 'trios_diffs_hs.txt' --path='DNM/trios/out/'